In [8]:
from typing import Literal, Optional, TypedDict
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, START, END

In [5]:
llm = ChatOllama(model = "gpt-oss:120b-cloud")

In [9]:
class AgentState(TypedDict):
    question: str
    selected_agent: Optional[str]
    answer: Optional[str]

In [10]:
def router_node(state: AgentState):
    question = state["question"].lower()

    technical_keywords = [
        "python",
        "code",
        "programming",
        "langchain",
        "langgraph",
        "api",
        "database",
        "error",
        "function",
        "class"
    ]

    if any(
        keyword in question
        for keyword in technical_keywords
    ):
        selected_agent = "technical"
    else:
        selected_agent = "general"

    return {
        "selected_agent": selected_agent
    }

In [11]:
def general_agent_node(state: AgentState):
    question = state["question"]

    prompt = f"""
You are a general knowledge assistant.

Answer the user's question in simple and clear language.

User question:
{question}
"""

    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )

    return {
        "answer": response.content
    }

In [12]:
def general_agent_node(state: AgentState):
    question = state["question"]

    prompt = f"""
You are a general knowledge assistant.

Answer the user's question in simple and clear language.

User question:
{question}
"""

    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )

    return {
        "answer": response.content
    }

In [15]:
def technical_agent_node(state: AgentState):
    question = state["question"]

    prompt = f"""
You are a senior software engineer.

Answer the technical question accurately.
Explain the concept step by step.
Include a small code example when useful.

User question:
{question}
"""

    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )

    return {
        "answer": response.content
    }

In [16]:
def route_to_agent(
    state: AgentState
) -> Literal["general_agent", "technical_agent"]:

    if state["selected_agent"] == "technical":
        return "technical_agent"

    return "general_agent"

In [24]:
graph_builder = StateGraph(AgentState)

graph_builder.add_node("router", router_node)
graph_builder.add_node("general_agent", general_agent_node)
graph_builder.add_node("technical_agent", technical_agent_node)

graph_builder.add_edge(START, "router")
graph_builder.add_conditional_edges(
    "router",
    route_to_agent,
    {
        "general_agent": "general_agent",
        "technical_agent": "technical_agent",
    },
)
graph_builder.add_edge("general_agent", END)
graph_builder.add_edge("technical_agent", END)

graph = graph_builder.compile()

result = graph.invoke(
    {
        "question": "What is a conditional edge in LangGraph?"
    }
)

print("Selected agent:", result["selected_agent"])
print()
print("Answer:")


NameError: name 'HumanMessage' is not defined

In [23]:
print(result["answer"])

NameError: name 'result' is not defined